# In Silico Optimization
While fast, gradient solvers have a fatal flaw in biology: they get stuck in local optima (fake, smaller peaks) and they do not understand uncertainty. To conquer complex bioprocess manifolds, the industry standard is Bayesian Optimization (BO).
## 🧭 The Bayesian Paradigm: Exploration vs. Exploitation
Bayesian Optimization does not just blindly step uphill. It acts like an intelligent scientist. It looks at the data, forms a hypothesis about where the peak might be, calculates how confident it is, and then deliberately chooses the next best experiment to run.\
It balances two competing desires:\
- Exploitation: Searching near known high-titer areas to push slightly higher.
- Exploration: Searching completely untested areas of the DoE space because the uncertainty is so high that a massive, undiscovered peak might exist there.
To do this, Bayesian Optimization relies on two mathematical pillars:

### Pillar 1: The Surrogate Model (Gaussian Processes)
Instead of fitting a rigid polynomial or a deterministic neural network, BO typically uses a Gaussian Process (GP) (also known as Kriging).\
A GP does not just predict the mean titer; it predicts a distribution of possible titers. For any given Temperature and pH, the GP outputs: "I predict the titer is 5.5 g/L, with a 95% confidence interval of $\pm$ 0.8 g/L."\
Where you have dense data, the uncertainty is near zero. Where you have no data, the uncertainty explodes.
### Pillar 2: The Acquisition Function
This is the mathematical formula that tells the algorithm exactly which bioreactor condition to test next. The most common is Expected Improvement (EI).\
EI looks at the Gaussian Process and calculates a score for every possible combination of Temperature and pH. It rewards conditions that have a high predicted titer and conditions that have high uncertainty. The algorithm finds the highest EI score, and that becomes your next physical laboratory experiment.
## 💻 Exercise: Coding the Bayesian Optimizer
Use of the `scikit-optimize` (`skopt`) library. This is how to set up an automated "Active Learning" loop to find the peak of our `true_bioreactor` function with the absolute minimum number of experiments.\
**Task:** Read how we define the boundaries of the biological space and let the Gaussian Process hunt for the peak.

In [1]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

In [2]:
def true_bioreactor(X, noise_sd=0.1):
    Temp, pH, FS, Gl, NaBu = X[:, 0], X[:, 1], X[:, 2], X[:, 3],X[:, 4]

    # The true peak is hidden at: Temp=36.8 °C, pH=6.8, Glucose=5.5 g/L (max titer ~6.5 g/L)
    # Feed Start (FS) and NaBu have tiny coefficients (noise)
    titer = (
    6.5 - 0.8*(Temp-36.8)**2 - 2.5*(pH-6.8)**2 - 0.4*(Gl-5.5)**2 + 0.6*(Temp-36.8)*(pH-6.8) + 0.01*FS - 0.03*NaBu
    )

    # Add Biological/assay noise
    noise = np.random.normal(0, noise_sd, size=len(Temp))

    # Titer cannot drop below 0 g/L in reality
    return np.maximum(0, titer + noise)

In [3]:
# 1. Define the Biological Design Space (The boundaries of our map)
# We tell the algorithm it can only search within these physical limits
space = [
    Real(31.0, 38.0, name='temp'),   # wide enough to contain the true optimum (36.8 °C)
    Real(6.6, 7.4, name='ph'),       # true optimum (6.8) sits inside, not on the edge
    Real(3.0, 8.0, name='glucose')
]
space

[Real(low=31.0, high=38.0, prior='uniform', transform='identity'),
 Real(low=6.6, high=7.4, prior='uniform', transform='identity'),
 Real(low=3.0, high=8.0, prior='uniform', transform='identity')]

In [4]:
# 2. Wrap our Biological Ground Truth into an Objective Function
# The decorator maps the 'space' dimensions to the function arguments
@use_named_args(space)
def objective(**params):
    # Retrieve the physical parameters
    T = params['temp']
    pH = params['ph']
    G = params['glucose']

    # We lock Feed Start at 4.0 and NaBu at 0.5 mM, the same values used in notebook 1, Phase 2
    X_input = np.array([[T, pH, 4.0, G, 0.5]])

    # Run the experiment (querying our true_bioreactor function from section 1).
    titer = true_bioreactor(X_input)[0]

    # Bayesian Optimization algorithms are built to MINIMIZE by default.
    # To maximize titer, we must return the negative titer.
    return -titer

In [5]:
# 3. Run the GP Optimizer
# n_calls=20 means the algorithm is only allowed to run 20 total "experiments"

np.random.seed(5)  # seeds the simulated assay noise, so results are reproducible

print("--- STARTING BAYESIAN OPTIMIZATION ---\n")
res = gp_minimize(
    func=objective,             # The function to minimize
    dimensions=space,           # The biological limits
    acq_func='EI',              # Expected improvement (balances exploration/exploitation)
    n_calls=20,                 # Budget: maximum number of bioreactors we can afford
    random_state=5              # Seed for reproducibility
)

# 4. Interpret the Results
print(f'Optimization Complete after {len(res.func_vals)} physical experiments.')
# res.fun is the NEGATIVE titer (we minimized -titer), so flip the sign back
print(f'Maximum Titer Found: {-res.fun:.3f} g/L (best observed run, includes assay noise)')
print(f'Optimal Parameters Found:')
print(f'    Temperature: {res.x[0]:.2f} °C')
print(f'    pH:          {res.x[1]:.2f}')
print(f'    Glucose:     {res.x[2]:.2f} g/L')

# 5. How close did 20 runs get to the true optimum? (noise-free evaluation, only possible in silico)
X_found = np.array([[res.x[0], res.x[1], 4.0, res.x[2], 0.5]])
X_true  = np.array([[36.8, 6.8, 4.0, 5.5, 0.5]])
titer_found = true_bioreactor(X_found, noise_sd=0)[0]
titer_max   = true_bioreactor(X_true,  noise_sd=0)[0]
print(f'\nTrue optimum: Temperature 36.80 °C, pH 6.80, Glucose 5.50 g/L -> {titer_max:.3f} g/L')
print(f'True titer at the conditions BO found: {titer_found:.3f} g/L '
      f'({100 * titer_found / titer_max:.1f}% of the achievable maximum with 20 experiments)')

--- STARTING BAYESIAN OPTIMIZATION ---



Optimization Complete after 20 physical experiments.
Maximum Titer Found: 6.396 g/L (best observed run, includes assay noise)
Optimal Parameters Found:
    Temperature: 37.19 °C
    pH:          6.60
    Glucose:     5.84 g/L

True optimum: Temperature 36.80 °C, pH 6.80, Glucose 5.50 g/L -> 6.525 g/L
True titer at the conditions BO found: 6.210 g/L (95.2% of the achievable maximum with 20 experiments)


### Reading the result
With a budget of 20 runs, BO reaches about 95% of the achievable titer. Temperature and glucose land within ~0.4 units of the true optimum. pH ends up at the lower edge of the search space (6.6) rather than at 6.8. That is not a bug: on its own, moving pH from 6.8 to 6.6 costs only ~0.1 g/L, the same size as the simulated assay noise (SD 0.1 g/L), so with 20 noisy measurements the GP cannot tell those conditions apart. The reported "maximum titer found" (6.40 g/L) is also higher than the true titer at those conditions (6.21 g/L): the best single noisy run is optimistic, which is why confirmation runs matter. In a real campaign, the next step would be a few replicate confirmation runs around pH 6.6 to 6.9 to resolve it.